# SCCR reproduction notebook

Reproduce the Storage Cost Coverage Ratio (SCCR) end to end from the frozen capture, then run the sensitivity and confidence-interval analysis.

**Formula (model-spec v2.1.0):**

```
R_blocks = 365.25 * 24 * 6              blocks/year
cb       = C / (B_block * R_blocks)     USD per byte per year
L_net    = B_block * cb * T * N         USD per block (network lifetime cost)
SCCR_i   = (fee_sats/1e8 * USD_i) / L_net
```

Everything comes from `research/model-spec.json` and the frozen capture — no constant is redefined here.

**Layer:** observed (frozen node/capture inputs) + modelled (assumption grid).

In [ ]:
import json, os, statistics, random

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
SPEC = os.path.join(REPO, 'research', 'model-spec.json')
CAP  = os.path.join(REPO, 'research', 'reproduce', 'input', 'fee_history_capture.json')

spec = json.load(open(SPEC)); q = spec['quantities']
C, N, T, B = q['C']['value'], q['N']['value'], q['T']['value'], q['B_block']['value']
capture = json.load(open(CAP))
print('spec', spec['version'], '| C', C, 'N', N, 'T', T, 'B_block', B, '| capture blocks', len(capture))

## 1. Compute the SCCR per block

In [ ]:
R = 365.25 * 24 * 6
cb = C / (B * R)
L_net = B * cb * T * N
ratios = []
for e in capture:
    fs = e.get('avgFees', 0); usd = e.get('USD') or 0
    if not fs: continue
    ratios.append(((fs/1e8)*usd)/L_net)
below = [r for r in ratios if r < 1.0]
print(f'L_net (USD/block)  : {L_net:.6f}')
print(f'cb (USD/byte/year) : {cb:.6e}')
print(f'blocks             : {len(ratios)}')
print(f'avg SCCR           : {statistics.mean(ratios):.6f}')
print(f'min / max          : {min(ratios):.6f} / {max(ratios):.6f}')
print(f'below 1.0          : {len(below)}/{len(ratios)} ({100*len(below)/len(ratios):.1f}%)')

## 2. Sensitivity grid — SCCR across N x T x C

SCCR is inverse-linear in N, T and C, so this is exact for the stated ranges.

In [ ]:
fee_usd = statistics.mean(ratios) * L_net
rows = []
for n in (10_000, 32_000, 100_000):
    for t in (5, 10, 20):
        for c in (500, 925, 1500):
            ln = n * t * c / R
            rows.append((n, t, c, fee_usd/ln))
vals = sorted(r[3] for r in rows)
print(f'combinations       : {len(rows)}')
print(f'SCCR band          : {vals[0]:.3f} - {vals[-1]:.3f} (median {statistics.median(vals):.3f})')
print(f'below 1.0          : {sum(1 for v in vals if v<1)}/{len(vals)}')
for n,t,c,s in sorted(rows, key=lambda x:x[3])[:3]: print(f'  lowest  N={n:,} T={t} C={c}: SCCR {s:.3f}')
for n,t,c,s in sorted(rows, key=lambda x:-x[3])[:3]: print(f'  highest N={n:,} T={t} C={c}: SCCR {s:.3f}')

## 3. Distribution and bootstrap 95% CI for the mean

From the real per-block ratios (a CI for the *sample mean*, not the population).

In [ ]:
sv = sorted(ratios); n = len(sv)
def pct(q_): return sv[min(n-1, max(0, round(q_*(n-1))))]
rnd = random.Random(20260917)
boot = sorted(sum(ratios[rnd.randrange(n)] for _ in range(n))/n for _ in range(10_000))
print(f'mean {statistics.mean(ratios):.6f}  median {statistics.median(ratios):.6f}')
print(f'IQR {pct(0.25):.6f} - {pct(0.75):.6f}   P5-P95 {pct(0.05):.6f} - {pct(0.95):.6f}')
print(f'bootstrap 95% CI for the mean: {boot[250]:.6f} - {boot[-250]:.6f}')

## 4. Interpretation

- The mean SCCR sits well below 1.0 and the CI does not approach 1.0.
- The assumption grid shows the index crosses 1.0 only at the corner most favourable to coverage (high N, long T, high C).
- **N (node count) dominates the uncertainty** — state it whenever the SCCR is quoted.

See `research/sccr-sensitivity.md` for the published version of this analysis.